# Fine-tune Laya for incident classification

**Before running:** Runtime → Change runtime type → **GPU** (T4 is fine, A100 is faster). Then *Runtime → Run all*.
When the upload box appears, choose **`laya_colab.zip`**. At the end, **`laya_ft.pt`** (~30 MB) downloads automatically — send it back.

What this does: loads Laya (ModernBERT-large encoder, pinned revision), adds LoRA adapters + 3 heads
(incident type / sentence relevance / form fields), trains on our 1,164 training conversations (one sample per user turn,
user text only), validates each epoch, calibrates confidence, and scores both the fine-tuned model and the
**un-fine-tuned zero-shot baseline** on the 128-conversation test set at full precision.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install "transformers==4.48.3" "huggingface_hub<1.0" tokenizers safetensors

In [ ]:
import os, zipfile
from google.colab import files
if not os.path.exists("laya_colab.zip"):
    files.upload()
zipfile.ZipFile("laya_colab.zip").extractall("work")
os.chdir("work"); print(os.listdir("."))

In [ ]:
import sys, json, time, math, copy, random
import numpy as np, torch, torch.nn.functional as F
sys.path.insert(0, ".")
from common import *
from huggingface_hub import snapshot_download
from tokenizers import Tokenizer

LD = snapshot_download(LAYA_REPO, revision=LAYA_REVISION, allow_patterns=LAYA_FILES)
tok = Tokenizer.from_file(os.path.join(LD, "tokenizer/tokenizer.json"))
tax, types, fields = label_space("incident_taxonomy.json")
train_c, val_c = train_val_split(load_conversations("out", "train"))
test_c = load_conversations("out", "test")
S = {"train": build_samples(train_c, tok, types, fields), "val": build_samples(val_c, tok, types, fields),
     "test": build_samples(test_c, tok, types, fields)}
print({k: len(v) for k, v in S.items()}, "| types", len(types), "| fields", len(fields))

In [ ]:
# ---- config
SEED, EPOCHS, BS, LR_LORA, LR_HEAD, R, ALPHA = 0, 6, 16, 3e-4, 1e-3, 16, 32
W_TYPE, W_REL, W_FIELD = 1.0, 0.5, 0.5
dev = "cuda"
use_bf16 = torch.cuda.is_bf16_supported()
amp_dtype = torch.bfloat16 if use_bf16 else torch.float16
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

model = LayaClassifier(add_lora(load_laya_encoder(LD, attn="sdpa"), R, ALPHA), len(types), len(fields)).to(dev)
lora_p = [p for n, p in model.named_parameters() if p.requires_grad and n.startswith("enc.")]
head_p = [p for n, p in model.named_parameters() if p.requires_grad and not n.startswith("enc.")]
opt = torch.optim.AdamW([{"params": lora_p, "lr": LR_LORA}, {"params": head_p, "lr": LR_HEAD}], weight_decay=0.01)
steps = EPOCHS * math.ceil(len(S["train"]) / BS)
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, s / (0.06 * steps)) * max(0.0, (steps - s) / steps))
scaler = torch.amp.GradScaler("cuda", enabled=not use_bf16)
print("trainable M:", round(sum(p.numel() for p in lora_p + head_p) / 1e6, 2), "| amp:", amp_dtype)

def losses(b, out):
    tl, rl, fl = out
    lt = F.cross_entropy(tl.float(), b["type"])
    lr = F.cross_entropy(rl.float().reshape(-1, 2), b["rel"].reshape(-1), ignore_index=-100)
    lf = (F.binary_cross_entropy_with_logits(fl.float(), b["fields"], reduction="none").mean(-1) * b["field_mask"]).sum() / b["field_mask"].sum().clamp(min=1)
    return W_TYPE * lt + W_REL * lr + W_FIELD * lf

@torch.no_grad()
def predict(m, samples, amp=True, bs=64):
    m.eval(); T, R_, Fl = [], [], []
    for i in range(0, len(samples), bs):
        b = {k: v.to(dev) for k, v in collate(samples[i:i + bs]).items()}
        with torch.autocast("cuda", dtype=amp_dtype, enabled=amp):
            t, r, f = m(b["input_ids"], b["attention_mask"])
        T.append(t.float().cpu()); R_.append(r.float().cpu()); Fl.append(f.float().cpu())
    return torch.cat(T).numpy(), torch.cat(R_).numpy(), torch.cat(Fl).numpy()

In [ ]:
best, best_state, t0 = -1, None, time.time()
g = torch.Generator().manual_seed(SEED)
for ep in range(EPOCHS):
    model.train(); order = torch.randperm(len(S["train"]), generator=g).tolist(); tot = 0
    for i in range(0, len(order), BS):
        b = {k: v.to(dev) for k, v in collate([S["train"][j] for j in order[i:i + BS]]).items()}
        with torch.autocast("cuda", dtype=amp_dtype):
            loss = losses(b, model(b["input_ids"], b["attention_mask"]))
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward(); scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(lora_p + head_p, 1.0)
        scaler.step(opt); scaler.update(); sched.step(); tot += float(loss)
    v = evaluate(S["val"], *predict(model, S["val"]), types=types)
    score = v["type_acc_all_turns"] + 0.5 * v["fields_micro_f1"]
    print(f"epoch {ep+1}/{EPOCHS} loss {tot / math.ceil(len(order) / BS):.3f} | val " +
          " ".join(f"{k}={v[k]:.3f}" for k in ("type_acc_all_turns", "type_acc_final_turn", "type_macro_f1", "relevance_acc", "fields_micro_f1")) +
          f" | {time.time() - t0:.0f}s")
    if score > best:
        best, best_state = score, copy.deepcopy(trainable_state(model))
model.load_state_dict(best_state, strict=False)
print("best val score", round(best, 4))

In [ ]:
# ---- temperature scaling for the type head (fit on validation, full precision)
vt, _, _ = predict(model, S["val"], amp=False)
vy = torch.tensor([s["type"] for s in S["val"]]); logT = torch.zeros(1, requires_grad=True)
o = torch.optim.LBFGS([logT], lr=0.1, max_iter=100)
def nll():
    o.zero_grad(); l = F.cross_entropy(torch.from_numpy(vt) / logT.exp(), vy); l.backward(); return l
o.step(nll); TEMP = float(logT.exp()); print("temperature", round(TEMP, 3))

In [ ]:
# ---- test set, full precision: fine-tuned
ft_test = evaluate(S["test"], *predict(model, S["test"], amp=False), types=types)
print("FINE-TUNED (fp32) :", json.dumps(ft_test, indent=1))

# ---- test set, full precision: un-fine-tuned Laya, zero-shot vs taxonomy descriptions
base = LayaEmbedder(load_laya_encoder(LD, attn="sdpa")).to(dev).eval()
protos = prototype_texts(tax)
with torch.no_grad():
    pi = [encode_text(tok, protos[t]) for t in types]
    P = base(torch.tensor(np.stack([a for a, _ in pi])).to(dev), torch.tensor(np.stack([b for _, b in pi])).to(dev))
    E = []
    for i in range(0, len(S["test"]), 64):
        b = collate(S["test"][i:i + 64])
        E.append(base(b["input_ids"].to(dev), b["attention_mask"].to(dev)))
    zs_logits = (torch.cat(E) @ P.T * 20).cpu().numpy()
zs_test = evaluate(S["test"], zs_logits, types=types)
print("BASE zero-shot (fp32):", json.dumps(zs_test, indent=1))

In [ ]:
out = {"state": best_state, "types": types, "fields": fields, "lora": {"r": R, "alpha": ALPHA}, "L": L,
       "temperature": TEMP, "laya_revision": LAYA_REVISION, "test_fp32_finetuned": ft_test,
       "test_fp32_zeroshot": zs_test, "config": {"epochs": EPOCHS, "bs": BS, "lr_lora": LR_LORA, "lr_head": LR_HEAD,
       "amp": str(amp_dtype), "gpu": torch.cuda.get_device_name()}}
torch.save(out, "/content/laya_ft.pt")
print(round(os.path.getsize("/content/laya_ft.pt") / 2**20, 1), "MB")
files.download("/content/laya_ft.pt")